# Ride-Hailing Method Comparison (Year 2023, Daily)

This notebook compares day-to-day ridership movement across four NYC ride-hailing services for the **full year 2023** at a **daily** level:
- FHV
- HVFHV
- Green Taxi
- Yellow Taxi

Outputs are saved to `data/02-processed/TLC_ridehail/analysis_yearly_daily`.
This notebook does **not** use hourly July outputs.


In [11]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, pearsonr, ttest_rel

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)


In [12]:
INPUT_DIR = Path('data/01-interim/TLC_ridehail')
OUTPUT_DIR = Path('data/02-processed/TLC_ridehail/analysis_yearly_daily')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SERVICE_FILES = {
    'fhv': '2023_For_Hire_Vehicles_Trip_Data_cleaned.csv',
    'hvfhv': '2023_High_Volume_FHV_Trip_Data_cleaned.csv',
    'green_taxi': '2023_Green_Taxi_Trip_Data_cleaned.csv',
    'yellow_taxi': '2023_Yellow_Taxi_Trip_Data_cleaned.csv',
}

OUTPUT_DIR


PosixPath('data/02-processed/TLC_ridehail/analysis_yearly_daily')

In [13]:
def find_col(df: pd.DataFrame, names: list[str]) -> str:
    m = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in m:
            return m[n.lower()]
    raise KeyError(f'Missing expected column from {names}')


def benjamini_hochberg(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    ranked = p_values[order]

    adjusted_ranked = np.empty(n, dtype=float)
    prev = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = ranked[i] * n / rank
        prev = min(prev, val)
        adjusted_ranked[i] = prev

    adjusted = np.empty(n, dtype=float)
    adjusted[order] = np.clip(adjusted_ranked, 0, 1)
    return adjusted


In [14]:
daily_parts = []

for service, filename in SERVICE_FILES.items():
    df = pd.read_csv(INPUT_DIR / filename)

    date_col = find_col(df, ['by_day_pickup_datetime', 'pickup_datetime', 'date'])
    trip_col = find_col(df, ['trip_count', 'ride_count', 'count'])

    temp = df[[date_col, trip_col]].copy()
    temp.columns = ['date', 'trip_count']

    temp['date'] = pd.to_datetime(temp['date'], errors='coerce')
    temp['trip_count'] = pd.to_numeric(
        temp['trip_count'].astype(str).str.replace(',', '', regex=False),
        errors='coerce',
    )

    temp = temp.dropna(subset=['date', 'trip_count'])
    temp['date'] = temp['date'].dt.normalize()

    daily_service = temp.groupby('date', as_index=False)['trip_count'].sum()
    daily_service = daily_service.rename(columns={'trip_count': service})

    daily_service.rename(columns={service: 'trip_count'}).to_csv(
        OUTPUT_DIR / f'2023_{service}_daily_trip_counts.csv', index=False
    )

    daily_parts.append(daily_service)

summary = pd.DataFrame({
    'service': list(SERVICE_FILES.keys()),
    'rows': [len(d) for d in daily_parts],
    'total_trips': [int(d[s].sum()) for d, s in zip(daily_parts, SERVICE_FILES.keys())],
})
summary


/var/folders/4q/r8lz6hr96qx82pxg02dggqx80000gn/T/ipykernel_14163/1363122573.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp['date'] = pd.to_datetime(temp['date'], errors='coerce')
/var/folders/4q/r8lz6hr96qx82pxg02dggqx80000gn/T/ipykernel_14163/1363122573.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp['date'] = pd.to_datetime(temp['date'], errors='coerce')
/var/folders/4q/r8lz6hr96qx82pxg02dggqx80000gn/T/ipykernel_14163/1363122573.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp['date'] = pd.to_datetime(temp['date'], errors='coerce')
/var/folde

,service,rows,total_trips
0,fhv,365,3524987
1,hvfhv,365,232490020
2,green_taxi,365,787053
3,yellow_taxi,365,38309612


In [15]:
daily = daily_parts[0]
for d in daily_parts[1:]:
    daily = daily.merge(d, on='date', how='outer')

daily = daily.sort_values('date').reset_index(drop=True)
daily.to_csv(OUTPUT_DIR / '2023_daily_totals_by_service.csv', index=False)

all_daily = (
    daily[['date'] + list(SERVICE_FILES.keys())]
    .set_index('date')
    .sum(axis=1)
    .rename('trip_count')
    .reset_index()
)
all_daily.to_csv(OUTPUT_DIR / '2023_all_ridehail_daily_trip_counts.csv', index=False)

daily.head()


,date,fhv,hvfhv,green_taxi,yellow_taxi
0,2023-01-01,4357,629154,1463,76189
1,2023-01-02,3750,408084,1564,65819
2,2023-01-03,7808,477829,2125,85733
3,2023-01-04,8578,494039,2383,94998
4,2023-01-05,8598,517870,2417,100960


## Statistical Comparison Setup

We compare services using daily log ratios:

$$ \log\left(\frac{\text{today trips}}{\text{yesterday trips}}\right) $$

Then we run:
- Friedman repeated-measures test (overall differences across services)
- Pairwise paired t-tests (mean difference in daily log change)
- Pairwise Pearson correlations (co-movement)


In [16]:
services = list(SERVICE_FILES.keys())
complete = daily.dropna(subset=services).copy()

ratios = np.log(complete[services] / complete[services].shift(1))
ratios = ratios.replace([np.inf, -np.inf], np.nan).dropna()

ratios_with_date = ratios.copy()
ratios_with_date.insert(0, 'date', complete['date'].iloc[1:].values)
ratios_with_date.head()


,date,fhv,hvfhv,green_taxi,yellow_taxi
1,2023-01-02,-0.150028,-0.432903,0.066758,-0.146309
2,2023-01-03,0.733393,0.157780,0.306525,0.264329
3,2023-01-04,0.094052,0.033362,0.114588,0.102618
4,2023-01-05,0.002329,0.047110,0.014167,0.060869
5,2023-01-06,-0.019140,0.160816,0.051996,0.011816


In [17]:
friedman = friedmanchisquare(*(ratios[s].values for s in services))
overall_test = pd.DataFrame([{
    'test': 'friedman_repeated_measures',
    'n_days': len(ratios),
    'statistic': float(friedman.statistic),
    'p_value': float(friedman.pvalue),
}])
overall_test


,test,n_days,statistic,p_value
0,friedman_repeated_measures,364,43.021978,2.434701e-09


In [18]:
pair_rows = []
for a, b in combinations(services, 2):
    t_res = ttest_rel(ratios[a], ratios[b], nan_policy='omit')
    diff = ratios[a] - ratios[b]
    pair_rows.append({
        'service_a': a,
        'service_b': b,
        'n_days': int(diff.notna().sum()),
        'mean_diff_log_ratio': float(diff.mean()),
        't_statistic': float(t_res.statistic),
        'p_value': float(t_res.pvalue),
    })

pairwise_ttests = pd.DataFrame(pair_rows)
pairwise_ttests['p_value_fdr_bh'] = benjamini_hochberg(pairwise_ttests['p_value'].to_numpy())
pairwise_ttests.sort_values('p_value_fdr_bh')


,service_a,service_b,n_days,mean_diff_log_ratio,t_statistic,p_value,p_value_fdr_bh
0,fhv,hvfhv,364,-0.000250,-0.015246,0.987844,0.995123
1,fhv,green_taxi,364,0.000052,0.006116,0.995123,0.995123
2,fhv,yellow_taxi,364,-0.000107,-0.006581,0.994753,0.995123
3,hvfhv,green_taxi,364,0.000301,0.029481,0.976497,0.995123
4,hvfhv,yellow_taxi,364,0.000143,0.011386,0.990922,0.995123
5,green_taxi,yellow_taxi,364,-0.000158,-0.013308,0.989389,0.995123


In [19]:
corr_rows = []
for a, b in combinations(services, 2):
    r, p = pearsonr(ratios[a], ratios[b])
    corr_rows.append({
        'service_a': a,
        'service_b': b,
        'n_days': len(ratios),
        'pearson_r': float(r),
        'p_value': float(p),
    })

pairwise_correlations = pd.DataFrame(corr_rows)
pairwise_correlations['p_value_fdr_bh'] = benjamini_hochberg(pairwise_correlations['p_value'].to_numpy())
pairwise_correlations.sort_values('pearson_r', ascending=False)


,service_a,service_b,n_days,pearson_r,p_value,p_value_fdr_bh
1,fhv,green_taxi,364,0.842904,1.811334e-99,1.086801e-98
5,green_taxi,yellow_taxi,364,0.322726,2.871661e-10,8.614983e-10
2,fhv,yellow_taxi,364,0.217575,2.825937e-05,4.238906e-05
4,hvfhv,yellow_taxi,364,0.145074,5.554113e-03,6.664935e-03
3,hvfhv,green_taxi,364,-0.089637,8.768882e-02,8.768882e-02
0,fhv,hvfhv,364,-0.227471,1.173051e-05,2.346103e-05


In [20]:
ratio_summary = ratios.describe().T.reset_index().rename(columns={'index': 'service'})

daily.to_csv(OUTPUT_DIR / '2023_daily_totals_by_service.csv', index=False)
ratios_with_date.to_csv(OUTPUT_DIR / '2023_daily_log_ratios_by_service.csv', index=False)
overall_test.to_csv(OUTPUT_DIR / '2023_overall_test.csv', index=False)
pairwise_ttests.to_csv(OUTPUT_DIR / '2023_pairwise_ttests.csv', index=False)
pairwise_correlations.to_csv(OUTPUT_DIR / '2023_pairwise_correlations.csv', index=False)
ratio_summary.to_csv(OUTPUT_DIR / '2023_ratio_summary.csv', index=False)

print('Saved yearly daily analysis outputs to', OUTPUT_DIR)
ratio_summary


Saved yearly daily analysis outputs to data/02-processed/TLC_ridehail/analysis_yearly_daily


,service,count,mean,std,min,25%,50%,75%,max
0,fhv,364.0,-0.000057,0.264684,-0.669081,-0.184720,-0.008265,0.082792,0.877923
1,hvfhv,364.0,0.000192,0.116318,-0.432903,-0.103801,0.044002,0.082325,0.252342
2,green_taxi,364.0,-0.000109,0.146551,-0.405826,-0.092998,0.008171,0.072632,0.537573
3,yellow_taxi,364.0,0.000049,0.227137,-2.230222,-0.067158,0.008260,0.065463,2.922043
